# Brute Force Approach for Minimum fuel Trajectories in Earth Moon System

This notebook applies a brute force approach to solve the problem of launching a rocket from Low Earth Orbit (LEO) to Low Moon Orbit (LMO).

Two impulsive burns are appied. One at LEO and one at LMO.

The time of flight and phase of departure are also optimized

### Imports

In [2]:
import numpy as np
import pandas as pd
from cr3bp import (
    create_earth_moon_system,
    grid_search_method
)

### Initialize the System / Problem

In [3]:
# Create the Earth-Moon system using the cr3bp module
em = create_earth_moon_system()
print(em.info())

CR3BP System Information:
  Primary 1 mass: 5.972e+24 kg
  Primary 2 mass: 7.342e+22 kg
  Primary 1 radius: 6.371e+06 m
  Primary 2 radius: 1.737e+06 m
  Total mass: 6.045e+24 kg
  Distance: 3.844e+08 m (384400.0 km)
  Mass parameter μ: 0.012145

Characteristic scales:
  Length (l*): 3.844e+08 m (384400.0 km)
  Time (t*): 3.752e+05 s (4.343 days)
  Velocity (v*): 1.025e+03 m/s (1.025 km/s)
  Acceleration (a*): 2.731e-03 m/s^2
  Period: 27.285 days
None


In [4]:
# Define the LEO and LMO altitudes in meters
leo_alt_m=463e3
lmo_alt_m=100e3

In [5]:
1e3 / em.l_star

2.6014568158168575e-06

## Run the Optimization Method

In [15]:
dec_var_ranges = [[3.80, 4.00],[2.90, 3.10], [0.10, 0.15], [0.6, 0.75]]
dec_var_ranges = new_ranges

In [ ]:
results_df = grid_search_method(em, dec_var_ranges, 2.6e-5, leo_alt_m, lmo_alt_m, print_intermediates=True)

Performing grid search with 20 x 20 x 10 x 20 = 80000 grid points...
Iteration: 14103/80000
Grid point satisfies constraint: Theta=3.90 rad, Delta_v=3.04, Delta_v_angle=0.13 rad, TOF=0.70 s => Distance to LMO=664.07 km, Total Delta_v=8.64 km/s
New optimal found: Delta_v=3.11 km/s, Theta=3.90 rad, Delta_v_angle=0.13 rad, TOF=0.70 s, Distance to LMO=664.07 km
Iteration: 14125/80000
Grid point satisfies constraint: Theta=3.90 rad, Delta_v=3.04, Delta_v_angle=0.14 rad, TOF=0.71 s => Distance to LMO=745.15 km, Total Delta_v=9.58 km/s
Iteration: 14382/80000
Grid point satisfies constraint: Theta=3.90 rad, Delta_v=3.04, Delta_v_angle=0.15 rad, TOF=0.70 s => Distance to LMO=97.84 km, Total Delta_v=6.44 km/s
New optimal found: Delta_v=3.12 km/s, Theta=3.90 rad, Delta_v_angle=0.15 rad, TOF=0.70 s, Distance to LMO=97.84 km
Iteration: 18104/80000
Grid point satisfies constraint: Theta=3.91 rad, Delta_v=3.04, Delta_v_angle=0.13 rad, TOF=0.70 s => Distance to LMO=560.80 km, Total Delta_v=7.90 km/s
I

In [17]:
print(results_df)

       theta   delta_v  delta_v_angle       tof  total_delta_v  \
0   3.918421  3.039474       0.126389  0.699515       3.474255   
1   3.939474  3.034211       0.123611  0.730367       3.790013   
2   3.934211  3.039474       0.145833  0.741586       4.313829   
3   3.944737  3.034211       0.131944  0.747195       4.319089   
4   3.918421  3.039474       0.129167  0.702320       4.459208   
5   3.923684  3.039474       0.137500  0.719148       4.488888   
6   3.934211  3.034211       0.120833  0.721953       5.449633   
7   3.918421  3.039474       0.134722  0.710734       5.804810   
8   3.902632  3.044737       0.145833  0.696711       6.288984   
9   3.913158  3.039474       0.131944  0.702320       7.196748   
10  3.907895  3.039474       0.134722  0.702320       7.715185   
11  3.913158  3.039474       0.145833  0.724758       7.718524   
12  3.923684  3.039474       0.143056  0.727562       7.791831   
13  3.918421  3.039474       0.140278  0.719148       7.796666   
14  3.9026

In [18]:
optimals = results_df.iloc[0][['theta', 'delta_v', 'delta_v_angle', 'tof']].values

In [19]:
np.save("grid_search_results.npy", optimals)

In [21]:
shrink = 0.5
new_ranges = []
for i in range(4):
    current_span = dec_var_ranges[i][1] - dec_var_ranges[i][0]
    half_width = current_span * shrink / 2
    new_min = max(dec_var_ranges[i][0], optimals[i] - half_width)
    new_max = min(dec_var_ranges[i][1], optimals[i] + half_width)
    new_ranges.append([new_min, new_max])
print(new_ranges)

[[np.float64(3.893421052631579), np.float64(3.943421052631579)], [np.float64(3.014473684210526), np.float64(3.064473684210526)], [np.float64(0.12083333333333333), np.float64(0.1326388888888889)], [np.float64(0.6967105263157894), np.float64(0.7128376038781163)]]


In [ ]:
results_df['distance_to_lmo'] = results_df['distance_to_lmo'] * em.l_star

In [27]:
print(results_df)

       theta   delta_v  delta_v_angle       tof  total_delta_v  \
0   3.918421  3.039474       0.126389  0.699515       3.474255   
1   3.939474  3.034211       0.123611  0.730367       3.790013   
2   3.934211  3.039474       0.145833  0.741586       4.313829   
3   3.944737  3.034211       0.131944  0.747195       4.319089   
4   3.918421  3.039474       0.129167  0.702320       4.459208   
5   3.923684  3.039474       0.137500  0.719148       4.488888   
6   3.934211  3.034211       0.120833  0.721953       5.449633   
7   3.918421  3.039474       0.134722  0.710734       5.804810   
8   3.902632  3.044737       0.145833  0.696711       6.288984   
9   3.913158  3.039474       0.131944  0.702320       7.196748   
10  3.907895  3.039474       0.134722  0.702320       7.715185   
11  3.913158  3.039474       0.145833  0.724758       7.718524   
12  3.923684  3.039474       0.143056  0.727562       7.791831   
13  3.918421  3.039474       0.140278  0.719148       7.796666   
14  3.9026